In [ ]:
!pip install -U bitsandbytes

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoModelForSeq2SeqLM, AutoTokenizer, pipeline, BitsAndBytesConfig
import pickle

**Change to false for similarity based examples:**

In [ ]:
use_cluster_examples = False

**Change to false for test set calculation:**

In [ ]:
use_validation = False

Load the correct file:

In [ ]:
if use_validation:
  if use_cluster_examples:
    with open("EXAMPLES_300_VAL_CLUSTER.pkl", "rb") as f:
      examples = pickle.load(f)
  else:
    with open("EXAMPLES_300_VAL_SIM.pkl", "rb") as f:
      examples = pickle.load(f)
else:
  if use_cluster_examples:
    with open("EXAMPLES_600_TEST_CLUSTER.pkl", "rb") as f:
      examples = pickle.load(f)
  else:
    with open("EXAMPLES_600_TEST_SIM.pkl", "rb") as f:
      examples = pickle.load(f)

## OUTPUTS FOR CONTRASTIVE LLAMA

In [ ]:
model_id_llama = "unsloth/llama-3-8b-Instruct-bnb-4bit"

model_llama = AutoModelForCausalLM.from_pretrained(
    model_id_llama,
    device_map="auto",
)

tokenizer_llama = AutoTokenizer.from_pretrained(model_id_llama, use_fast=True)

In [ ]:
pipe = pipeline(
    "text-generation",
    model=model_llama,
    tokenizer=tokenizer_llama,
    return_full_text=False
)

In [ ]:
def generate(instruction, system_prompt, pipe=pipe):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": instruction},
    ]

    prompt = tokenizer_llama.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    generation_parameters = {
        "max_new_tokens": 100,
        "do_sample": False,
        "return_full_text": False,
    }

    output = pipe(prompt, **generation_parameters)
    return output[0]["generated_text"]

300 examples - ~20 minutes <br>
600 examples - ~40 minutes

In [ ]:
results = []
system_prompt = "You are a helpful summarization assistant."

for example in examples:
    instruction = (
        "Summarize the following article in one sentence."
        "Do not preface the summary with anything.\n"
        f"Article: {example["doc"]}"
        )
    model_sum = generate(instruction, system_prompt)
    results.append({
        "inst_id": example["inst_id"],
        "doc_id": example["doc_id"],
        "doc": example["doc"],
        "llama_sum": model_sum
        })

In [ ]:
if use_validation:
  if use_cluster_examples:
    with open("LLAMA_300_VAL_CLUSTER.pkl", "wb") as f:
      pickle.dump(results, f)
  else:
    with open("LLAMA_300_VAL_SIM.pkl", "wb") as f:
      pickle.dump(results, f)
else:
  if use_cluster_examples:
    with open("LLAMA_600_Test_CLUSTER.pkl", "wb") as f:
      pickle.dump(results, f)
  else:
    with open("LLAMA_600_Test_SIM.pkl", "wb") as f:
      pickle.dump(results, f)

In [ ]:
#results[599]